In [1]:
import torch
import torch.nn as nn
import numpy as np
import glob
import pandas as pd
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
from torchvision import transforms

In [2]:
"""
  dataset class for the images, constructed from csv files containing pixel values of the preprocessed images
"""
class WordSet(Dataset):
  def __init__(self, state):
    dset = np.random.random((1, 128*32))
    self.labels = []
    if state == "train":
      path = "drive/MyDrive/word_detect/files/file*"
    else:
      path = "drive/MyDrive/word_detect/files/test/file*"
    for filename in glob.glob(path):
      df = pd.read_csv(filename)
      dset = np.vstack((dset, np.array(df.iloc[:, 1:-1], dtype = "float32")))
      self.labels += df.iloc[:, -1].tolist()
    dset = dset[1:]
    self.tensor = torch.from_numpy(dset).reshape((len(dset), 1, 32, 128)).to(dtype = torch.float32)
    mean, std = self.tensor.mean(), self.tensor.std()
    transform = transforms.Normalize((mean,), (std,))
    self.tensor = transform(self.tensor)


  def __len__(self):
    return self.tensor.shape[0]
  def __getitem__(self, ndx):
    return self.tensor[ndx], self.labels[ndx]

In [ ]:
def Dataset1():
    dset = np.random.random((1, 128*32))
    labels = []
    for filename in glob.glob("drive/MyDrive/word_detect/files/*"):
      df = pd.read_csv(filename)
      dset = np.vstack((dset, np.array(df.iloc[:, 1:-1], dtype = "float32")))
      labels += df.iloc[:, -1].tolist()
      tensor = torch.from_numpy(dset).reshape((len(dset), 1, 32, 128)).to(dtype = torch.float32)
      a = collate_fn
      return tensor, *collate_fn(labels)

In [ ]:
test = Dataset1()

In [3]:
class CRNN(nn.Module):
  def  __init__(self, charno):
    super(CRNN, self).__init__()
    self.charno = charno
    self.model1 = nn.Sequential(nn.Conv2d(1, 32, (3,3), padding = "same"),
                          nn.ReLU(),
                          nn.MaxPool2d((2,2)),

                          nn.Conv2d(32, 64, (3,3), padding = "same"),
                          nn.ReLU(),
                          nn.MaxPool2d((2,2)),
                          nn.Dropout2d(0.25),

                          nn.Conv2d(64, 128, (3,3), padding = "same"),
                          nn.ReLU(),
                          nn.MaxPool2d((2, 1)),
                          nn.Dropout2d(0.25),

                          nn.Conv2d(128, 256, (3,3), padding = "same"),
                          nn.ReLU(),
                          nn.MaxPool2d((2, 1)),
                          nn.Dropout2d(0.25),

                          nn.Conv2d(256, 256, (3,3), padding = "same"),
                          nn.ReLU(),
                          nn.MaxPool2d((2, 1)),
                          nn.Dropout2d(0.25)).to(dtype = torch.float32)
    self.model2 = nn.Sequential(nn.Linear(256, 32),
                                nn.LSTM(32, 128, bidirectional=True, batch_first=True)).to(dtype = torch.float32)
    self.model3 = nn.LSTM(256, 128, bidirectional=True, batch_first=True).to(dtype = torch.float32)
    self.model4 = nn.Sequential(nn.Linear(256, self.charno),
                                nn.Softmax(dim = 1))
  def forward(self, x):
    out = self.model1(x)
    batch_size, channels, height, width = out.size()
    out = out.permute(0,2,1,3).view(batch_size, width, -1)
    out, _ = self.model2(out)
    out, _ = self.model3(out)
    out = self.model4(out)
    return out


In [ ]:
characters = " ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz"
char_to_idx = {char: idx for idx, char in enumerate(characters)}

def encode_label(label, char_to_idx):
  return [char_to_idx[char] for char in label]

def collate_fn(labels):

    encoded_labels = [encode_label(label, char_to_idx) for label in labels]

    labels_flat = torch.cat([torch.tensor(label, dtype=torch.int32) for label in encoded_labels])

    label_lengths = torch.tensor([len(label) for label in encoded_labels], dtype=torch.int32)

    input_lengths = torch.full(size=(len(images),), fill_value=32, dtype=torch.int32)

    return labels_flat, label_lengths

In [ ]:
# Define the key components
model = CRNN(len(characters))  # Your CRNN model
criterion = nn.CTCLoss(blank=0)  # Assuming blank label is 0
optimizer = optim.Adam(model.parameters(), lr=0.001)

train = WordSet("train")
test = WordSet("test")
trainloader = DataLoader(train, batch_size=32, shuffle=True)
testloader = DataLoader(test)

# Training loop
num_epochs = 10
for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0

    for batch_idx, (images, labels, label_lengths) in enumerate(trainloader):


        # Forward pass
        logits = model(images)  # Shape: (batch_size, 32, num_classes)
        log_probs = nn.functional.log_softmax(logits, dim=2)  # Apply log softmax
        log_probs = log_probs.permute(1, 0, 2)  # (seq_len, batch_size, num_classes)

        # Compute input lengths (all are 32 in this case)
        input_lengths = torch.full(
            size=(log_probs.size(1),),  # batch_size
            fill_value=log_probs.size(0),  # seq_length (32)
            dtype=torch.long,
        )

        # Compute CTC Loss
        loss = criterion(log_probs, labels, input_lengths, label_lengths)

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Track loss
        epoch_loss += loss.item()

        if batch_idx % 10 == 0:
            print(f"Epoch [{epoch+1}/{num_epochs}], Batch [{batch_idx+1}], Loss: {loss.item():.4f}")

    test_images, test_labels, test_label_lengths = enumerate(testloader)[0]
    logits = model(test_images)  # Shape: (batch_size, 32, num_classes)
    log_probs = nn.functional.log_softmax(logits, dim=2)  # Apply log softmax
    log_probs = log_probs.permute(1, 0, 2)  # (seq_len, batch_size, num_classes)
    test_input_lengths = torch.full(
            size=(log_probs.size(1),),  # batch_size
            fill_value=log_probs.size(0),  # seq_length (32)
            dtype=torch.long,
        )

    test_loss = criterion(log_probs, test_labels, test_input_lengths, test_label_lengths)

    print(f"Epoch [{epoch+1}/{num_epochs}] Average Loss: {epoch_loss / len(trainloader):.4f}")
    print(f"\tAverage Loss: {test_loss / len(testloader):.4f}")


KeyError: tensor([[ 0.6422,  0.6422,  0.2629,  ...,  0.6422,  0.6422,  0.6422],
        [ 0.6422,  0.6422,  0.1474,  ...,  0.6257,  0.6257,  0.6422],
        [ 0.6422,  0.6092,  0.1969,  ...,  0.6257,  0.6422,  0.6422],
        ...,
        [ 0.3948, -0.7102, -1.9967,  ..., -1.9802, -1.1555, -0.3309],
        [ 0.6257,  0.4443,  0.0155,  ..., -0.1495,  0.3948,  0.4938],
        [ 0.6422,  0.6422,  0.5927,  ...,  0.5762,  0.6257,  0.6257]])

In [ ]:
a = model1()

In [ ]:
a.model1.forward(test[0]).shape

torch.Size([1001, 256, 1, 32])

In [ ]:
a.forward(test[0]).shape

torch.Size([1001, 32, 256])

In [ ]:
out = model.forward(test[0])

In [ ]:
out.shape

torch.Size([1001, 256, 1, 32])

In [ ]:
a = torch.from_numpy(np.random.rand(32,128)).to(dtype = torch.float32, device=torch.device('cuda'))
model2 = nn.Sequential(nn.Linear(128, 32),
                       nn.LSTM(32, 128, bidirectional=True, batch_first=True),
                       nn.LSTM(256, 128, bidirectional=True, batch_first=True)).to(device = torch.device('cuda'), dtype = torch.float32)

In [ ]:
a = torch.from_numpy(np.random.rand(32,128)).to(dtype = torch.float32, device=torch.device('cuda'))
b = nn.Linear(128, 32).to(dtype = torch.float32, device=torch.device('cuda'))
c = nn.LSTM(32, 64, bidirectional=True, batch_first=True).to(dtype = torch.float32, device=torch.device('cuda'))
c.forward(b.forward(a))[0].shape

torch.Size([32, 128])